# Training

---

### Libraries

In [197]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

---

### Loading dataset

In [198]:
df = pd.read_csv("../data/data.csv")

In [199]:
df["event_time"] = pd.to_datetime(df["event_time"], format="ISO8601")
df = df.sort_values(["account_id", "event_time"]).reset_index(drop=True)

In [200]:
print(df.shape)

(206368, 6)


In [201]:
df.head()

,account_id,amount,event_time,is_home,is_domestic,is_fraud
0,ML_00000,815000,2026-06-24 12:52:15.521757+00:00,True,True,False
1,ML_00000,838000,2026-06-24 13:27:06.843143+00:00,True,True,False
2,ML_00000,974000,2026-06-24 18:54:56.766292+00:00,True,True,False
3,ML_00000,1036000,2026-06-25 10:56:52.600864+00:00,True,True,False
4,ML_00002,24558000,2026-06-24 12:53:20.284844+00:00,True,True,False


---

### Feature Engineering

In [202]:
results = []
for _, grp in df.groupby("account_id"):
    grp = grp.sort_values("event_time").set_index("event_time")
    grp["avg_amount_24h"]         = grp["amount"].rolling("24h", closed="left", min_periods=1).mean()
    grp["tx_count_1h"]            = grp["amount"].rolling("1h",  closed="left", min_periods=0).count()
    grp["tx_count_3h"]            = grp["amount"].rolling("3h",  closed="left", min_periods=0).count()
    grp["time_since_last_tx_sec"] = grp.index.to_series().diff().dt.total_seconds()
    results.append(grp.reset_index())

df = pd.concat(results, ignore_index=True)

df["hour_of_day"]  = df["event_time"].dt.hour
df["amount_ratio"] = df["amount"] / df["avg_amount_24h"]

In [203]:
df[["account_id", "amount", "avg_amount_24h", "amount_ratio",
    "tx_count_1h", "tx_count_3h", "time_since_last_tx_sec", "hour_of_day", "is_fraud"]].head(10)

,account_id,amount,avg_amount_24h,amount_ratio,tx_count_1h,tx_count_3h,time_since_last_tx_sec,hour_of_day,is_fraud
0,ML_00000,815000,NaN,NaN,0.0,0.0,NaN,12,False
1,ML_00000,838000,8.150000e+05,1.028221,1.0,1.0,2091.321386,13,False
2,ML_00000,974000,8.265000e+05,1.178463,0.0,0.0,19669.923149,18,False
3,ML_00000,1036000,8.756667e+05,1.183099,0.0,0.0,57715.834572,10,False
4,ML_00002,24558000,NaN,NaN,0.0,0.0,NaN,12,False
5,ML_00002,26835000,2.455800e+07,1.092719,0.0,1.0,4543.869606,14,False
6,ML_00002,30898000,2.569650e+07,1.202421,0.0,2.0,4666.217437,15,False
7,ML_00002,29915000,2.743033e+07,1.090581,0.0,0.0,19211.944116,20,False
8,ML_00002,31711000,2.805150e+07,1.130456,0.0,0.0,45092.245816,9,False
9,ML_00002,19873000,2.878340e+07,0.690433,0.0,1.0,7325.517771,11,False


In [204]:
FEATURES = [
    "amount", "hour_of_day", "is_home", "is_domestic",
    "avg_amount_24h", "amount_ratio", "tx_count_1h", "tx_count_3h",  
    "time_since_last_tx_sec",
]

---

### Cleaning data

In [205]:
df["avg_amount_24h"]         = df["avg_amount_24h"].fillna(df["amount"])
df["amount_ratio"]           = df["amount_ratio"].fillna(1.0)
df["time_since_last_tx_sec"] = df["time_since_last_tx_sec"].fillna(0.0)

In [206]:
df["is_home"]     = df["is_home"].astype(int)
df["is_domestic"] = df["is_domestic"].astype(int)

In [207]:
df[["account_id", "amount", "avg_amount_24h", "amount_ratio",
    "tx_count_1h", "tx_count_3h", "time_since_last_tx_sec", "hour_of_day", "is_fraud"]].head(10)

,account_id,amount,avg_amount_24h,amount_ratio,tx_count_1h,tx_count_3h,time_since_last_tx_sec,hour_of_day,is_fraud
0,ML_00000,815000,8.150000e+05,1.000000,0.0,0.0,0.000000,12,False
1,ML_00000,838000,8.150000e+05,1.028221,1.0,1.0,2091.321386,13,False
2,ML_00000,974000,8.265000e+05,1.178463,0.0,0.0,19669.923149,18,False
3,ML_00000,1036000,8.756667e+05,1.183099,0.0,0.0,57715.834572,10,False
4,ML_00002,24558000,2.455800e+07,1.000000,0.0,0.0,0.000000,12,False
5,ML_00002,26835000,2.455800e+07,1.092719,0.0,1.0,4543.869606,14,False
6,ML_00002,30898000,2.569650e+07,1.202421,0.0,2.0,4666.217437,15,False
7,ML_00002,29915000,2.743033e+07,1.090581,0.0,0.0,19211.944116,20,False
8,ML_00002,31711000,2.805150e+07,1.130456,0.0,0.0,45092.245816,9,False
9,ML_00002,19873000,2.878340e+07,0.690433,0.0,1.0,7325.517771,11,False


---

### Label Noise

In [208]:
# RELABEL_SEED = 99
# rng = np.random.default_rng(RELABEL_SEED)

# def _z(s: pd.Series) -> pd.Series:
#     return (s - s.mean()) / (s.std() + 1e-8)

In [209]:
# is_night = df["hour_of_day"].between(1, 4).astype(float)  # 1h–4h

# risk_score = (
#     1.2 * _z(df["amount_ratio"])                    # amount >> avg account
#     + 0.2 * _z(df["tx_count_1h"])                   # velocity cao
#     + 0.2 * _z(df["tx_count_3h"])                   # velocity cao
#     + 0.3 * (1 - df["is_home"].astype(float))       # giao dịch ngoài nhà
#     + 0.6 * (1 - df["is_domestic"].astype(float))   # nước ngoài
#     + 0.4 * is_night                                # giờ đêm bất thường
#     + 2.5 * df["is_fraud"].astype(float)            # prior từ hard label
#     + rng.normal(0, 0.5, len(df))                   # noise 
# )

# prob_fraud    = 1 / (1 + np.exp(-(risk_score - 4)))
# df["is_fraud"] = rng.binomial(1, prob_fraud)

In [210]:
# n_fraud = df["is_fraud"].sum()
# print(f"After relabeling : {n_fraud:,} fraud  ({n_fraud/len(df)*100:.2f}%)")
# print(f"  → Nếu >3%: tăng shift  |  Nếu <2%: giảm shift")
# print(f"  → Nếu AUC > 0.95 sau train: tăng noise_std")

In [211]:
# df["is_fraud"] = df["is_fraud"].astype(bool) 

---

### Spliting train/ test dataset

In [212]:
fraud_accounts  = df[df["is_fraud"]]["account_id"].unique()
normal_accounts = df[~df["is_fraud"]]["account_id"].unique()

In [213]:
rng = np.random.default_rng(42)
fraud_accounts  = np.array(fraud_accounts)
normal_accounts = np.array(normal_accounts)
rng.shuffle(fraud_accounts)
rng.shuffle(normal_accounts)

In [214]:
n_fraud_train  = int(len(fraud_accounts)  * 0.8)
n_normal_train = int(len(normal_accounts) * 0.8)

In [215]:
train_accounts = set(np.concatenate([fraud_accounts[:n_fraud_train],
                                     normal_accounts[:n_normal_train]]))
test_accounts  = set(np.concatenate([fraud_accounts[n_fraud_train:],
                                     normal_accounts[n_normal_train:]]))

In [216]:
FEATURES = [
    "amount", "hour_of_day", "is_home", "is_domestic",
    "avg_amount_24h", "amount_ratio", "tx_count_1h", "tx_count_3h",  
    "time_since_last_tx_sec",
]

In [217]:
train = df[df["account_id"].isin(train_accounts)]
test  = df[df["account_id"].isin(test_accounts)]

X_train, y_train = train[FEATURES], train["is_fraud"].astype(int)
X_test,  y_test  = test[FEATURES],  test["is_fraud"].astype(int)

In [218]:
print(f"Train: {len(train):,} rows  |  fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Test : {len(test):,} rows   |  fraud: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

Train: 166,412 rows  |  fraud: 4,626 (2.78%)
Test : 42,549 rows   |  fraud: 1,713 (4.03%)


---

### Training model

In [219]:
scale_pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())

In [220]:
def objective(trial):
    params = {
        "n_estimators"         : 300,
        "max_depth"            : trial.suggest_int("max_depth", 3, 5),  # ← giới hạn xuống 5
        "learning_rate"        : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"            : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree"     : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight"     : trial.suggest_int("min_child_weight", 3, 15),  # ← tăng min
        "scale_pos_weight"     : scale_pos_weight,
        "eval_metric"          : "aucpr",
        "early_stopping_rounds": 20,
        "random_state"         : 42,
        "reg_alpha"            : trial.suggest_float("reg_alpha", 0.1, 2.0),   # ← L1
        "reg_lambda"           : trial.suggest_float("reg_lambda", 1.0, 5.0),  # ← L2
    }
    m = XGBClassifier(**params)
    m.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_prob = m.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, y_prob)

In [221]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

Best trial: 36. Best value: 0.99973: 100%|██████████| 50/50 [00:15<00:00,  3.14it/s] 


In [222]:
best = study.best_params
best.update({
    "n_estimators"         : 1000,
    "scale_pos_weight"     : scale_pos_weight,
    "eval_metric"          : "aucpr",
    "early_stopping_rounds": 20,
    "random_state"         : 42,
})

model = XGBClassifier(**best)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50,
)
print(f"Best iteration : {model.best_iteration}")

[0]	validation_0-aucpr:0.90798	validation_1-aucpr:0.91209
[50]	validation_0-aucpr:0.99987	validation_1-aucpr:0.99963
[87]	validation_0-aucpr:0.99995	validation_1-aucpr:0.99971
Best iteration : 67


---

### Evaluate

In [223]:
y_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

auc_roc = roc_auc_score(y_test, y_prob)
auc_pr  = average_precision_score(y_test, y_prob)

# Chọn threshold tối đa hóa F1
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx = f1s.argmax()
best_threshold = float(thresholds[best_idx])

y_pred = (y_prob >= best_threshold).astype(int)
p  = precision_score(y_test, y_pred)
r  = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Threshold : {best_threshold:.4f}\n")
print(f"Precision : {(p):.4f}  {'OK' if p > 0.80 else 'FAIL'}  (target > 0.80)")
print(f"Recall    : {(r):.4f}  {'OK' if r > 0.75 else 'FAIL'}  (target > 0.75)")
print(f"F1-Score  : {(f1):.4f}  {'OK' if f1 > 0.78 else 'FAIL'}  (target > 0.78)")
print(f"AUC-ROC   : {(auc_roc):.4f}  {'OK' if auc_roc > 0.85 else 'FAIL'}  (target > 0.85)")
print(f"AUC-PR    : {(auc_pr):.4f}  {'OK' if auc_pr > 0.70 else 'FAIL'}  (target > 0.70)")


Threshold : 0.9688

Precision : 0.9199  OK  (target > 0.80)
Recall    : 0.9264  OK  (target > 0.75)
F1-Score  : 0.9345  OK  (target > 0.78)
AUC-ROC   : 0.9393  OK  (target > 0.85)
AUC-PR    : 0.9093  OK  (target > 0.70)


---

### Save Model

In [225]:
out_path = Path("../data/fraud_detection_model.json")
out_path.parent.mkdir(exist_ok=True)

model.get_booster().save_model(str(out_path))

print(f"Saved : {out_path}")
print(f"Size  : {out_path.stat().st_size / 1024:.1f} KB")
print(f"Threshold dùng trong Flink: {best_threshold:.4f}")

Saved : ../data/fraud_detection_model.json
Size  : 116.3 KB
Threshold dùng trong Flink: 0.9688
